# 02 — Modelagem & Avaliação

**Runtime: GPU (T4)** no Colab. Carrega o dataset processado pequeno (gerado no notebook 01)
e treina os modelos. É a única parte que usa GPU — e é rápida/barata.

Tarefas cobertas: **T4** (modelos clássicos) · **T5** (1D-CNN) · **T6** (avaliação & comparação).

> Status e detalhes de cada tarefa: `docs/PLANO_IMPLEMENTACAO.md`.

In [ ]:
# ============================================================
#  SETUP — funciona no Google Colab e localmente
# ============================================================
import os, sys, subprocess
from pathlib import Path

# 1) Detecta o ambiente
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

# 2) Instala as libs que não vêm por padrão no Colab
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightkurve", "astroquery"], check=False)

# 3) Diretório do projeto (no Colab fica no Drive para persistir o cache)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/stellar-classification")
else:
    cwd = Path.cwd()
    PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

# 4) Caminhos de dados (criados se não existirem)
DATA_DIR      = PROJECT_DIR / "data"
CATALOG_DIR   = DATA_DIR / "catalogs"
RAW_LC_DIR    = DATA_DIR / "raw_lightcurves"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR   = PROJECT_DIR / "results"
for d in (CATALOG_DIR, RAW_LC_DIR, PROCESSED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# 5) Config global do projeto
CLASSES = ["binaria_eclipsante", "pulsante", "modulador_rotacional", "transito_exoplaneta"]
N_BINS = 512          # tamanho do vetor após phase-folding + binning (D4)
N_PER_CLASS = 1000    # alvo de amostras por classe (D3)
RANDOM_STATE = 42

print("PROJECT_DIR:", PROJECT_DIR)
print("Pastas de dados prontas em:", DATA_DIR)

In [ ]:
# Verifica a GPU (este notebook deve usar runtime GPU no Colab: Ambiente de execução > Alterar tipo)
try:
    import torch
    print("PyTorch — CUDA disponível:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch não instalado (a decisão de framework é da tarefa T5).")

## Carregar dataset processado
Lê `X.npy`, `y.npy`, `ids.npy` e `features.csv` de `PROCESSED_DIR`.

In [ ]:
# ============================================================
#  Carrega o dataset processado (gerado no notebook 01)
# ============================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X        = np.load(PROCESSED_DIR / "X.npy")        # (N, 512) — entrada da CNN
y        = np.load(PROCESSED_DIR / "y.npy")        # rótulo inteiro (índice em CLASSES)
ids      = np.load(PROCESSED_DIR / "ids.npy")      # TIC de cada amostra
features = pd.read_csv(PROCESSED_DIR / "features.csv")   # entrada dos modelos clássicos
SHORT = ["Binária", "Pulsante", "Rotacional", "Exoplaneta"]   # nomes curtos (ordem = CLASSES)

print("X:", X.shape, "| NaN:", int(np.isnan(X).sum()))
print("y por classe:", {SHORT[i]: int(c) for i, c in enumerate(np.bincount(y))})
print("features:", features.shape, "| alinhado:", len(features) == len(X) == len(y))

# Split treino/teste COMPARTILHADO (mesmos índices p/ T4 clássico e T5 CNN -> comparação justa no T6)
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"split compartilhado: treino={len(idx_tr)}  teste={len(idx_te)}")

## T4 — Modelos clássicos (Random Forest + Árvore + KNN + SVM)

Quatro classificadores clássicos sobre as **features físicas** de `features.csv`, com
`RobustScaler` + `StratifiedKFold(5)` + `GridSearchCV` (`scoring='f1_macro'`) + `class_weight='balanced'`.

⚠️ **Anti-vazamento (data leakage):** excluímos `used_ls` e `n_points`, que codificam a *origem*
do dado (catálogo/cadência) e não a física — usá-las deixaria o modelo "adivinhar" a classe pela
proveniência. Ficam **10 features físicas**: `log_period, ls_power, amp_p95_p05, amp_minmax, std,
mad, skew, kurt, depth, frac_below_half`.

Reporta Accuracy/Precision/Recall/F1-macro + recall da classe exoplaneta, no **conjunto de teste**
(split estratificado 80/20). Gera matrizes de confusão, importância de features (RF) e uma árvore ilustrativa.

In [ ]:
# ============================================================
#  T4 — treina RF, Árvore, KNN, SVM (features físicas)
# ============================================================
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

features["log_period"] = np.log10(features["period"].clip(lower=1e-3))
FEATS = ["log_period", "ls_power", "amp_p95_p05", "amp_minmax", "std",
         "mad", "skew", "kurt", "depth", "frac_below_half"]          # 10 features físicas (sem leakage)
Xf = features[FEATS].values
Xtr, Xte, ytr, yte = Xf[idx_tr], Xf[idx_te], y[idx_tr], y[idx_te]
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

def _pipe(clf): return Pipeline([("sc", RobustScaler()), ("clf", clf)])
GRIDS = {
    "RandomForest": (_pipe(RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
                     {"clf__n_estimators": [300, 500], "clf__max_depth": [None, 20], "clf__min_samples_leaf": [1, 2]}),
    "ArvoreDecisao": (_pipe(DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
                      {"clf__max_depth": [6, 10, None], "clf__min_samples_leaf": [1, 5, 10]}),
    "KNN": (_pipe(KNeighborsClassifier()),
            {"clf__n_neighbors": [5, 11, 21], "clf__weights": ["uniform", "distance"]}),
    "SVM_RBF": (_pipe(SVC(class_weight="balanced", random_state=RANDOM_STATE)),
                {"clf__C": [1, 10, 100], "clf__gamma": ["scale", 0.1]}),
}

clf_rows, best_clf = [], {}
for name, (pp, grid) in GRIDS.items():
    gs = GridSearchCV(pp, grid, scoring="f1_macro", cv=skf, n_jobs=-1).fit(Xtr, ytr)
    yp = gs.predict(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(yte, yp, average="macro", zero_division=0)
    rec_exo = precision_recall_fscore_support(yte, yp, labels=[3], average=None, zero_division=0)[1][0]
    clf_rows.append(dict(modelo=name, cv_f1=round(gs.best_score_, 4), acuracia=round(accuracy_score(yte, yp), 4),
                         precisao=round(pr, 4), recall=round(rc, 4), f1_macro=round(f1, 4), recall_exo=round(rec_exo, 4)))
    best_clf[name] = gs.best_estimator_
    print(f"  {name:14s} f1_macro={f1:.3f}  acc={accuracy_score(yte, yp):.3f}  recall_exo={rec_exo:.3f}")

clf_results = pd.DataFrame(clf_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
clf_results.to_csv(RESULTS_DIR / "t4_metrics.csv", index=False)
champ = clf_results.iloc[0]["modelo"]
print("\n=== desempenho (teste) ===\n", clf_results.to_string(index=False))
print(f"\nMelhor clássico: {champ}\n")
print(classification_report(yte, best_clf[champ].predict(Xte), target_names=SHORT, zero_division=0))

# matrizes de confusão (normalizadas por linha = recall)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, name in zip(axes.ravel(), GRIDS):
    cm = confusion_matrix(yte, best_clf[name].predict(Xte), normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1); ax.set_title(name)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(SHORT, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(SHORT, fontsize=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_ylabel("verdadeiro"); ax.set_xlabel("previsto")
fig.suptitle("T4 — matrizes de confusão (teste)", fontsize=12)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t4_confusion_matrices.png", dpi=130); plt.show()

# importância de features (RF) + árvore ilustrativa
imp = pd.Series(best_clf["RandomForest"].named_steps["clf"].feature_importances_, index=FEATS).sort_values()
fig, ax = plt.subplots(figsize=(8, 5)); ax.barh(imp.index, imp.values, color="#4C72B0")
ax.set_title("T4 — importância das features (Random Forest)"); ax.set_xlabel("importância")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t4_feature_importance.png", dpi=130); plt.show()

viz = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=RANDOM_STATE).fit(Xf, y)
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(viz, feature_names=FEATS, class_names=SHORT, filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title("T4 — árvore de decisão ilustrativa (profundidade 3)")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t4_decision_tree.png", dpi=130); plt.show()

## T5 — Modelo 1D-CNN (PyTorch)

Rede convolucional 1D sobre a curva dobrada `X` (512 pontos, representação `arcsinh` que preserva profundidade).

- **Arquitetura:** 3 blocos `Conv1d(1→32→64→128, kernel=5)` + `BatchNorm` + `LeakyReLU` + `MaxPool1d(2)` → `Flatten` → `Linear(128)` + `BatchNorm` + `Dropout(0.4)` → `Linear(4)`.
- **Treino:** `CrossEntropyLoss` com **pesos de classe** (desbalanceamento), `Adam` (lr 1e-3), **early stopping** na F1-macro de validação.
- **Split idêntico ao T4** (mesma seed) → comparação justa no T6. Roda em GPU se disponível, senão CPU (dataset pequeno, ~30 s).

> ⚠️ O `BatchNorm` + a entrada `arcsinh` (em vez de z-score, que apagava a amplitude) foram decisivos: levaram a CNN de F1≈0,61 para ≈0,82.

In [ ]:
# ============================================================
#  T5 — treina a 1D-CNN (PyTorch)
# ============================================================
import time
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dev)

# split interno treino/val (a partir do treino compartilhado); teste = idx_te (igual ao T4)
idx_t, idx_v = train_test_split(idx_tr, test_size=0.15, stratify=y[idx_tr], random_state=RANDOM_STATE)
def _tens(ix): return (torch.tensor(X[ix][:, None, :], dtype=torch.float32),
                       torch.tensor(y[ix], dtype=torch.long))
Xt, yt = _tens(idx_t); Xv, yv = _tens(idx_v); Xe, ye = _tens(idx_te)

class CNN1D(nn.Module):
    def __init__(self, n=4):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2),  nn.BatchNorm1d(32),  nn.LeakyReLU(0.1), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.LeakyReLU(0.1), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.MaxPool1d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128 * 64, 128), nn.BatchNorm1d(128),
                                  nn.LeakyReLU(0.1), nn.Dropout(0.4), nn.Linear(128, n))
    def forward(self, x): return self.head(self.feat(x))

model = CNN1D().to(dev)
counts = np.bincount(y[idx_t])
cw = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=dev)  # pesos de classe
crit = nn.CrossEntropyLoss(weight=cw)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
dl = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)

best_f1, best_state, wait, patience, hist = -1, None, 0, 12, []
t0 = time.perf_counter()
for ep in range(100):
    model.train()
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        vlogit = model(Xv.to(dev)); vloss = crit(vlogit, yv.to(dev)).item()
        vp = vlogit.argmax(1).cpu().numpy()
    _, _, vf1, _ = precision_recall_fscore_support(yv.numpy(), vp, average="macro", zero_division=0)
    hist.append((ep, float(loss.item()), vloss, vf1))
    if vf1 > best_f1:
        best_f1, wait = vf1, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= patience: break
print(f"treino: {ep+1} épocas, {time.perf_counter()-t0:.0f}s, melhor val f1-macro={best_f1:.3f}")

model.load_state_dict(best_state); model.eval()
with torch.no_grad():
    cnn_pred = model(Xe.to(dev)).argmax(1).cpu().numpy()       # previsões no teste (p/ o T6)
yte_cnn = ye.numpy()
acc = accuracy_score(yte_cnn, cnn_pred)
pr, rc, f1, _ = precision_recall_fscore_support(yte_cnn, cnn_pred, average="macro", zero_division=0)
rec_exo = precision_recall_fscore_support(yte_cnn, cnn_pred, labels=[3], average=None, zero_division=0)[1][0]
cnn_row = dict(modelo="1D-CNN", cv_f1=round(best_f1, 4), acuracia=round(acc, 4),
               precisao=round(pr, 4), recall=round(rc, 4), f1_macro=round(f1, 4), recall_exo=round(rec_exo, 4))
torch.save(best_state, RESULTS_DIR / "t5_cnn_state.pt")
print(f"\n=== 1D-CNN (teste) === acc={acc:.3f} f1-macro={f1:.3f} recall_exo={rec_exo:.3f}\n")
print(classification_report(yte_cnn, cnn_pred, target_names=SHORT, zero_division=0))

# figura: curva de treino + matriz de confusão
H = np.array(hist)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.plot(H[:, 0], H[:, 1], label="treino loss"); a1.plot(H[:, 0], H[:, 2], label="val loss")
a1.set_xlabel("época"); a1.set_ylabel("loss"); a1.legend(loc="upper right"); a1.set_title("1D-CNN — perda")
a1b = a1.twinx(); a1b.plot(H[:, 0], H[:, 3], color="green", ls="--"); a1b.set_ylabel("val f1-macro", color="green")
cm = confusion_matrix(yte_cnn, cnn_pred, normalize="true")
a2.imshow(cm, cmap="Blues", vmin=0, vmax=1); a2.set_title(f"1D-CNN — confusão (teste) f1={f1:.2f}")
a2.set_xticks(range(4)); a2.set_yticks(range(4))
a2.set_xticklabels(SHORT, rotation=45, ha="right", fontsize=8); a2.set_yticklabels(SHORT, fontsize=8)
for i in range(4):
    for j in range(4):
        a2.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
a2.set_xlabel("previsto"); a2.set_ylabel("verdadeiro")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t5_cnn_training.png", dpi=130); plt.show()

## T6 — Avaliação consolidada & comparação

Junta os 5 modelos (4 clássicos + 1D-CNN) numa **tabela comparativa única** (Acc/Prec/Rec/F1-macro + recall de exoplaneta),
um gráfico de barras de F1-macro, e as **matrizes de confusão lado a lado** (melhor clássico × CNN) no mesmo conjunto de teste.
Tudo salvo em `results/` para o artigo (Seção IV).

In [ ]:
# ============================================================
#  T6 — tabela comparativa + gráficos (usa resultados do T4 e T5)
# ============================================================
from matplotlib.patches import Patch

comp = pd.concat([clf_results, pd.DataFrame([cnn_row])], ignore_index=True)
comp = comp.sort_values("f1_macro", ascending=False).reset_index(drop=True)
comp["tipo"] = np.where(comp["modelo"] == "1D-CNN", "Deep Learning", "Clássico")
comp.to_csv(RESULTS_DIR / "t6_comparison.csv", index=False)
print("=== TABELA COMPARATIVA FINAL (teste) ===")
print(comp[["modelo", "acuracia", "precisao", "recall", "f1_macro", "recall_exo", "tipo"]].to_string(index=False))
print("\nMelhor modelo geral:", comp.iloc[0]["modelo"])

# gráfico de barras: F1-macro por modelo
order = comp.sort_values("f1_macro")
colors = ["#DD8452" if t == "Deep Learning" else "#4C72B0" for t in order["tipo"]]
fig, ax = plt.subplots(figsize=(9, 4.6))
bars = ax.barh(order["modelo"], order["f1_macro"], color=colors)
for b, (_, r) in zip(bars, order.iterrows()):
    ax.text(r["f1_macro"] + 0.005, b.get_y() + b.get_height()/2,
            f"{r['f1_macro']:.3f}  (rec.exo {r['recall_exo']:.2f})", va="center", fontsize=9)
ax.set_xlim(0, 1.25); ax.set_xlabel("F1-macro (teste)"); ax.set_title("Comparação de modelos (4 classes)")
ax.legend(handles=[Patch(color="#4C72B0", label="Clássico"), Patch(color="#DD8452", label="Deep Learning")],
          loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t6_comparison.png", dpi=140); plt.show()

# matrizes de confusão lado a lado: melhor clássico × CNN (mesmos índices de teste)
yte_idx = y[idx_te]
clf_pred = best_clf[champ].predict(Xf[idx_te])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, pred) in zip(axes, [(f"Melhor clássico: {champ}", clf_pred), ("1D-CNN", cnn_pred)]):
    cm = confusion_matrix(yte_idx, pred, normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1); ax.set_title(title)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(SHORT, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(SHORT, fontsize=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
    ax.set_xlabel("previsto"); ax.set_ylabel("verdadeiro")
fig.suptitle("T6 — matrizes de confusão (mesmo conjunto de teste)", fontsize=12)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "t6_confusion_best.png", dpi=130); plt.show()